# Hiệu năng pipeline

## Phạm vi đo

Các benchmark được nộp đều sử dụng chạy tiếp. Bảng trình bày từng phiên cùng phạm vi và trạng thái, không cộng thời gian các phiên chồng lặp và không so sánh tốc độ giữa các thành viên như những thí nghiệm tương đương. Hai phiên TV4 bổ sung ngày 04/11 là phiên trước lượt cuối, có số đo nhưng chưa có snapshot thống kê hoàn tất.

Thông tin CPU-only lấy từ môi trường benchmark tự ghi nhận; không suy việc hoàn tất thu thập chỉ từ việc notebook đã chạy tới cell benchmark.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.to_html(index=False, escape=True, border=0, classes='dataframe',
                              formatters={column:number for column in frame.columns})))


In [ ]:
rows = []
for s in sessions:
    b = s['benchmark']
    rows.append({'Phiên': s['session'], 'Phạm vi': run_range(b),
                 'Wall time (giây)': b['wall_seconds'],
                 'Kết thúc': 'Bị ngắt' if b['exit_code']==130 else ('Exit 0' if b['exit_code']==0 else 'Có lỗi (exit '+str(b['exit_code'])+')'),
                 'Số mẫu RAM': b['sample_count']})
table(rows)

## Thời gian thực thi

Wall time là giá trị được lưu cho từng lượt đo, không phải tổng thời gian đã dùng để thu thập dải cá nhân. Khoảng thời gian giữa timestamp bắt đầu/kết thúc có thể lớn hơn trường wall time; cần xác minh biên đo và overhead trước khi dùng làm benchmark tổng cuối cùng. Không ngoại suy thời gian của run sáu ID thành thời gian 5.000 ID.

Bảng dưới được tính trực tiếp từ các event đã hoàn tất. Thời gian từng paper có thể bao gồm thời gian stage; không cộng các loại event thành tổng runtime. Event đang dở lúc gián đoạn có thể chưa được ghi.

In [ ]:
timing_rows = []
for s in sessions:
    path = s['directory']/'timings.jsonl'
    events = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    discovery = [e for e in events if e['kind']=='discovery']
    papers = [e for e in events if e['kind']=='paper']
    timing_rows.append({'Phiên': s['session'],
        'Discovery: tổng giây event': sum(e['elapsed_seconds'] for e in discovery) if discovery else None,
        'Số event paper hoàn tất': len(papers),
        'Paper: trung bình giây/event': sum(e['elapsed_seconds'] for e in papers)/len(papers) if papers else None,
        'Event dùng cache': sum(e.get('cached') is True for e in events)})
table(timing_rows)

## Bộ nhớ và dung lượng đĩa

Peak RAM là cực đại RSS quan sát trên cây tiến trình; mean RAM là trung bình có trọng số thời gian. Phiên không có mẫu hoặc chỉ có một mẫu và không có khoảng quan sát được ghi **không đủ mẫu**. Disk là logical bytes của output root; dữ liệu các phiên trước có thể đã tồn tại, nên final disk không phải số byte riêng phiên đó tải mới.

In [ ]:
rows = []
for s in sessions:
    b=s['benchmark']; ram=b['ram']; disk=b['disk']
    usable=b['sample_count']>=2 and ram['observed_seconds']>0
    rows.append({'Phiên': s['session'],
        'Peak RAM (MiB)': ram['peak_rss_bytes']/1024**2 if usable and ram['peak_rss_bytes'] is not None else 'Không đủ mẫu',
        'Mean RAM (MiB)': ram['time_weighted_mean_rss_bytes']/1024**2 if usable and ram['time_weighted_mean_rss_bytes'] is not None else 'Không đủ mẫu',
        'Peak disk (GiB)': disk['peak_observed_bytes']/1024**3 if disk.get('peak_observed_bytes') is not None else None,
        'Final disk (GiB)': disk['final_bytes']/1024**3 if disk.get('final_bytes') is not None else None})
table(rows)
chosen = next(s for s in sessions if s['member']=='TV3')
samples=[json.loads(line) for line in (chosen['directory']/'samples.jsonl').read_text().splitlines() if line.strip()]
if len(samples)>1:
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot([v['elapsed_seconds']/60 for v in samples],[v['rss_bytes']/1024**2 for v in samples])
    ax.set_xlabel('Thời gian lấy mẫu (phút)');ax.set_ylabel('Tổng RSS (MiB)')
    ax.set_title('RAM quan sát — riêng lượt resume TV3 đã nộp')
    plt.tight_layout();plt.show()

Biểu đồ chỉ minh họa diễn biến trong một phiên có đủ mẫu, không đại diện toàn bộ quá trình thu thập hoặc phép so sánh giữa người chạy. Số mẫu thực tế quan trọng hơn interval cấu hình: thao tác trên Drive và quét disk có thể làm khoảng lấy mẫu dài hơn dự kiến.

## Kết luận về hiệu năng

Bằng chứng hiện có mô tả được các lượt chạy thực tế, chi phí xử lý/chờ và biến động tài nguyên ở những phiên có mẫu. Chưa đủ cơ sở xác nhận benchmark toàn assignment đồng nhất hoặc tổng thời gian thu thập toàn nhóm. Run ID, môi trường và thời điểm của từng phiên được lưu ở [phụ lục tái lập](audit-appendix.ipynb).